In [14]:
from __future__ import annotations
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display





# ---------------------------
# Config
# ---------------------------
RESULTS_ROOT_CANDIDATES = [Path("results/baselines"), Path("../results/baselines")]
BASELINE_AGENT = "random_generator_baseline"

POLICY_MAP = {
    "random_generator_baseline": ("Random", "Random", "Random"),
    "random_generator_diversity_planner_comp": ("Random", "Diversity", "Random"),
    "random_generator_llm_planner": ("Random", "LLM", "Random"),
    "chemeleon_generative_baseline": ("Chemeleon", "Random", "Random"),
    "chemeleon_diversity_planner_comp": ("Chemeleon", "Diversity", "Random"),
    "chemeleon_llm_planner": ("Chemeleon", "LLM", "Random"),
    "chemeleon_mlip_ranking_chain_filter": ("Chemeleon", "-", "MLIP"),
    "llm_react_orchestrator": ("LLM Orch.", "-", "-"),
}

# Display decimals for value(error)
METRIC_SPECS = {
    "AF": ("AF_mean", "AF_sem", 2),
    "EF": ("EF_mean", "EF_sem", 2),
    "AUDC": ("AUDC_mean", "AUDC_sem", 3),
    "mSUN": ("mSUN_mean", "mSUN_sem", 3),
    "Mean Comp. L1.": ("mean_comp_l1_mean", "mean_comp_l1_sem", 2),
    "Unique Comps.": ("unique_comps_mean", "unique_comps_sem", 1),
    "Unique SGs": ("unique_sgs_mean", "unique_sgs_sem", 2),
}
REPO_ROOT = "/home/weiyong/repos/MADE"
POLICY_RUN_DIRS = {
    "random_generator_baseline": [
        
        Path("/home/weiyong/repos/MADE/results/baselines/20260225-203807/random_generator_baseline_systems_ternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/20260226-070605/random_generator_baseline_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/20260226-183241/random_generator_baseline_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV")
    ],
    "llm_orch_qwen30b": [
        Path("/home/weiyong/repos/MADE/results/baselines/20260228-215337/llm_react_orchestrator_systems_ternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/20260302-020653/llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"), 
        Path("/home/weiyong/repos/MADE/results/baselines/20260303-210738/llm_react_orchestrator_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV")
        
    ],
    "llm_orch_qwen30b_reflx": [
        Path("/home/weiyong/repos/MADE/results/baselines/fidelity/qwen3-30b-instr_reflx/systems_ternary_n10_maxatoms20_intermetallic_smact/20260312-131554/llm_react_orchestrator_systems_ternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/fidelity/qwen3-30b-instr_reflx/systems_quaternary_n10_maxatoms20_intermetallic_smact/20260312-131554/llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/fidelity/qwen3-30b-instr_reflx/systems_quinary_n10_maxatoms20_intermetallic_smact/20260312-131554/llm_react_orchestrator_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV")
    ], 
    "llm_orch_qwen122b_reflx": [
        Path("/home/weiyong/repos/MADE/results/baselines/balanced/qwen3dot5-122b_reflx/systems_ternary_n10_maxatoms20_intermetallic_smact/20260312-130728/llm_react_orchestrator_systems_ternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"),
        Path("/home/weiyong/repos/MADE/results/baselines/balanced/qwen3dot5-122b_reflx/systems_quaternary_n10_maxatoms20_intermetallic_smact/20260312-130728/llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV"), 
        Path("/home/weiyong/repos/MADE/results/baselines/balanced/qwen3dot5-122b_reflx/systems_quinary_n10_maxatoms20_intermetallic_smact/20260312-130728/llm_react_orchestrator_systems_quinary_n10_maxatoms20_intermetallic_smact_10systems_50queries_100stabilitymeV")
    ]
    
}





# ---------------------------
# Helpers
# ---------------------------
def sem(values):
    s = pd.Series(values, dtype="float64").dropna()
    if len(s) <= 1:
        return 0.0
    return float(s.std(ddof=1) / math.sqrt(len(s)))

def fmt_table1(mean, sem_val, decimals):
    if pd.isna(mean):
        return "-"
    sem_val = 0.0 if pd.isna(sem_val) else float(sem_val)
    bracket = int(round(sem_val * (10 ** decimals)))
    return f"{float(mean):.{decimals}f}({bracket})"

def get_results_root():
    root = next((p for p in RESULTS_ROOT_CANDIDATES if p.exists()), None)
    if root is None:
        raise FileNotFoundError("Could not find results/baselines (or ../results/baselines).")
    return root

def extract_metrics_history(trajectory_blob):
    fes = trajectory_blob.get("final_env_state")
    if isinstance(fes, dict):
        mh = fes.get("metrics_history")
        if isinstance(mh, list) and mh:
            return mh

    # Fallback reconstruction from step-level observations
    history = []
    cumulative = 0
    for idx, obs in enumerate(trajectory_blob.get("trajectory", []), start=1):
        if isinstance(obs, dict) and obs.get("is_newly_discovered") and obs.get("is_stable"):
            cumulative += 1
        history.append({"queries_used": idx, "num_newly_discovered_stable": cumulative})
    return history

def curve_from_history(history):
    q, d = [], []
    for item in history:
        if not isinstance(item, dict):
            continue
        qq = item.get("queries_used")
        dd = item.get("num_newly_discovered_stable")
        if qq is None or dd is None:
            continue
        qq, dd = float(qq), float(dd)
        if np.isfinite(qq) and np.isfinite(dd):
            q.append(qq)
            d.append(dd)

    if not q:
        return np.array([]), np.array([])

    q = np.asarray(q, dtype=float)
    d = np.asarray(d, dtype=float)

    order = np.argsort(q)
    q, d = q[order], d[order]

    uq, idx = np.unique(q, return_index=True)
    ud = d[idx]

    if uq[0] > 0:
        uq = np.concatenate([[0.0], uq])
        ud = np.concatenate([[0.0], ud])

    return uq, ud

def enhancement_factor(proposal_history, baseline_history, cap=100.0):
    pq, pdv = curve_from_history(proposal_history)
    bq, bdv = curve_from_history(baseline_history)
    if len(pq) == 0 or len(bq) == 0:
        return 0.0
    final_q = pq[-1]
    final_p = pdv[-1]
    baseline_at_final_q = float(np.interp(final_q, bq, bdv, left=0.0, right=bdv[-1]))
    baseline_safe = max(baseline_at_final_q, 1e-8)
    return float(min(final_p / baseline_safe, cap))

def acceleration_factor(proposal_history, baseline_history):
    pq, pdv = curve_from_history(proposal_history)
    bq, bdv = curve_from_history(baseline_history)
    if len(pq) == 0 or len(bq) == 0:
        return 0.0

    target = bdv[-1]  # same default target used in MADE metrics
    p_hits = np.where(pdv >= target)[0]
    b_hits = np.where(bdv >= target)[0]

    p_needed = float("inf") if len(p_hits) == 0 else float(pq[p_hits[0]])
    b_needed = float("inf") if len(b_hits) == 0 else float(bq[b_hits[0]])

    if p_needed == 0 and b_needed == 0:
        return 1.0
    if p_needed == 0:
        return float("inf")
    if np.isinf(p_needed) or np.isinf(b_needed):
        return 0.0
    return float(b_needed / p_needed)

def to_markdown_table(df):
    cols = list(df.columns)
    lines = [
        "| " + " | ".join(cols) + " |",
        "| " + " | ".join(["---"] * len(cols)) + " |",
    ]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(lines)


# ---------------------------
# 1) Explicitly select run dirs per agent
# ---------------------------


run_records = []

for agent, exp_dirs in POLICY_RUN_DIRS.items():
    for exp_dir in exp_dirs:
        exp_dir = Path(exp_dir)

        summary_path = exp_dir / "overall_summary" / "summary.json"
        meta_path = exp_dir / "experiment_metadata.json"
        progress_path = exp_dir / "progress.json"

        if not summary_path.exists():
            print(f"[skip] missing summary.json: {summary_path}")
            continue
        if not meta_path.exists():
            print(f"[skip] missing experiment_metadata.json: {meta_path}")
            continue
        if not progress_path.exists():
            print(f"[skip] missing progress.json: {progress_path}")
            continue

        meta = json.loads(meta_path.read_text())
        progress = json.loads(progress_path.read_text())
        summary_blob = json.loads(summary_path.read_text())

        m = re.search(
            r"systems_(ternary|quaternary|quinary)",
            meta.get("systems_file", "") or exp_dir.name
        )
        system_size = m.group(1) if m else "unknown"

        expected_episodes = (meta.get("max_systems") or 0) * (meta.get("num_episodes") or 0)

        run_records.append(
            {
                "run_timestamp": exp_dir.parent.name,
                "agent": agent,
                "system_size": system_size,
                "experiment_dir": exp_dir,
                "completed": progress.get("status") == "completed",
                "episodes_total": summary_blob.get("episodes_total", 0),
                "expected_episodes": expected_episodes,
            }
        )

runs = pd.DataFrame(run_records)
if runs.empty:
    raise RuntimeError("No runs found from POLICY_RUN_DIRS.")

runs["is_valid"] = runs["completed"] & (runs["episodes_total"] == runs["expected_episodes"])
selected_runs = runs[runs["is_valid"]].reset_index(drop=True)

if selected_runs.empty:
    raise RuntimeError("No valid completed runs found in POLICY_RUN_DIRS.")

    
# ---------------------------
# 2) Load episode metrics + trajectories
# ---------------------------
episode_rows = []
histories_by_agent = {}

for run in selected_runs.itertuples(index=False):
    exp_dir = Path(run.experiment_dir)
    agent = run.agent
    size = run.system_size

    for episodes_path in sorted(exp_dir.glob("systems/*/summary/episodes.json")):
        system = episodes_path.parents[1].name
        episodes = json.loads(episodes_path.read_text())
        for ep in episodes:
            episode_rows.append(
                {
                    "agent": agent,
                    "system_size": size,
                    "system": system,
                    "episode_id": int(ep.get("episode_id", 0)),
                    "AUDC": ep.get("final/discovery_curve_area_under_discovery_curve_normalized", np.nan),
                    "mSUN": ep.get("final/novelty_stable_unique_novel_fraction", np.nan),
                    "Mean Comp. L1.": ep.get("final/diversity_all_composition_l1_distance_mean", np.nan),
                    "Unique Comps.": ep.get("final/novelty_stable_unique_novel_count", np.nan),
                    "Unique SGs": ep.get("final/diversity_all_structure_unique_spacegroups_count", np.nan),
                }
            )

    for traj_path in sorted(exp_dir.glob("systems/*/trajectories/episode_*.json")):
        system = traj_path.parents[1].name
        episode_id = int(traj_path.stem.split("_")[-1])
        traj = json.loads(traj_path.read_text())
        hist = extract_metrics_history(traj)
        histories_by_agent.setdefault(agent, {})[(size, system, episode_id)] = hist

episodes_df = pd.DataFrame(episode_rows)
if episodes_df.empty:
    raise RuntimeError("No episodes loaded from selected runs.")

# ---------------------------
# 3) Aggregate metrics over ALL sizes/systems/episodes
# ---------------------------
agg = (
    episodes_df.groupby("agent")
    .agg(
        AUDC_mean=("AUDC", "mean"),
        AUDC_sem=("AUDC", sem),
        mSUN_mean=("mSUN", "mean"),
        mSUN_sem=("mSUN", sem),
        mean_comp_l1_mean=("Mean Comp. L1.", "mean"),
        mean_comp_l1_sem=("Mean Comp. L1.", sem),
        unique_comps_mean=("Unique Comps.", "mean"),
        unique_comps_sem=("Unique Comps.", sem),
        unique_sgs_mean=("Unique SGs", "mean"),
        unique_sgs_sem=("Unique SGs", sem),
    )
    .reset_index()
)

if BASELINE_AGENT not in histories_by_agent:
    raise RuntimeError(f"Baseline agent '{BASELINE_AGENT}' not found in selected runs.")

baseline_histories = histories_by_agent[BASELINE_AGENT]
afef_rows = []

for agent, agent_histories in histories_by_agent.items():
    if agent == BASELINE_AGENT:
        n = len(baseline_histories)
        af_vals = [1.0] * n
        ef_vals = [1.0] * n
    else:
        af_vals, ef_vals = [], []
        for key, proposal_history in agent_histories.items():
            base_history = baseline_histories.get(key)
            if base_history is None:
                continue
            af = acceleration_factor(proposal_history, base_history)
            ef = enhancement_factor(proposal_history, base_history)
            if np.isfinite(af):
                af_vals.append(af)
            if np.isfinite(ef):
                ef_vals.append(ef)

    afef_rows.append(
        {
            "agent": agent,
            "AF_mean": float(np.mean(af_vals)) if af_vals else np.nan,
            "AF_sem": sem(af_vals),
            "EF_mean": float(np.mean(ef_vals)) if ef_vals else np.nan,
            "EF_sem": sem(ef_vals),
        }
    )

afef_df = pd.DataFrame(afef_rows)
res = agg.merge(afef_df, on="agent", how="left")

# Policy columns + row ordering
res[["Generator", "Planner", "Selector"]] = res["agent"].apply(
    lambda a: pd.Series(POLICY_MAP.get(a, (a, "-", "-")))
)
order_map = {k: i for i, k in enumerate(POLICY_MAP.keys())}
res["row_order"] = res["agent"].map(order_map).fillna(9999)
res = res.sort_values(["row_order", "agent"]).reset_index(drop=True)

# ---------------------------
# 4) Table-1-style formatted output
# ---------------------------
disp = res[["Generator", "Planner", "Selector"]].copy()

for metric_name, (mean_col, sem_col, decimals) in METRIC_SPECS.items():
    best = res[mean_col].max(skipna=True)
    values = []
    for mean_val, sem_val in zip(res[mean_col], res[sem_col]):
        txt = fmt_table1(mean_val, sem_val, decimals)
        if pd.notna(mean_val) and np.isclose(mean_val, best, atol=1e-12, rtol=0.0):
            txt = f"**{txt}**"
        values.append(txt)
    disp[metric_name] = values

runs_preview = selected_runs[["run_timestamp", "agent", "system_size", "episodes_total", "expected_episodes"]].copy()

md = []
md.append("### Table 1-style MADE Results")
md.append("Averaged across all selected system sizes and episodes. Higher is better. Values are `mean(error)` where `error` is SEM in the last shown digits.")
md.append("**Policy:** Generator / Planner / Selector  \n**Discovery Performance:** AF, EF, AUDC, mSUN  \n**Discovery Diversity:** Mean Comp. L1., Unique Comps., Unique SGs")
md.append("#### Runs Used (latest valid per agent/size)")
md.append(to_markdown_table(runs_preview))
md.append("#### Results")
md.append(to_markdown_table(disp))

display(Markdown("\n\n".join(md)))


### Table 1-style MADE Results

Averaged across all selected system sizes and episodes. Higher is better. Values are `mean(error)` where `error` is SEM in the last shown digits.

**Policy:** Generator / Planner / Selector  
**Discovery Performance:** AF, EF, AUDC, mSUN  
**Discovery Diversity:** Mean Comp. L1., Unique Comps., Unique SGs

#### Runs Used (latest valid per agent/size)

| run_timestamp | agent | system_size | episodes_total | expected_episodes |
| --- | --- | --- | --- | --- |
| 20260225-203807 | random_generator_baseline | ternary | 50 | 50 |
| 20260226-070605 | random_generator_baseline | quaternary | 50 | 50 |
| 20260226-183241 | random_generator_baseline | quinary | 50 | 50 |
| 20260228-215337 | llm_orch_qwen30b | ternary | 50 | 50 |
| 20260302-020653 | llm_orch_qwen30b | quaternary | 50 | 50 |
| 20260303-210738 | llm_orch_qwen30b | quinary | 30 | 30 |
| 20260312-131554 | llm_orch_qwen30b_reflx | ternary | 50 | 50 |
| 20260312-131554 | llm_orch_qwen30b_reflx | quaternary | 50 | 50 |
| 20260312-131554 | llm_orch_qwen30b_reflx | quinary | 50 | 50 |
| 20260312-130728 | llm_orch_qwen122b_reflx | ternary | 50 | 50 |
| 20260312-130728 | llm_orch_qwen122b_reflx | quaternary | 50 | 50 |
| 20260312-130728 | llm_orch_qwen122b_reflx | quinary | 50 | 50 |

#### Results

| Generator | Planner | Selector | AF | EF | AUDC | mSUN | Mean Comp. L1. | Unique Comps. | Unique SGs |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Random | Random | Random | 1.00(0) | 1.00(0) | 0.117(9) | 0.118(10) | **0.94(1)** | 5.9(5) | 1.00(0) |
| llm_orch_qwen122b_reflx | - | - | 5.43(67) | **12.89(209)** | **0.391(19)** | **0.375(17)** | 0.78(2) | **18.8(9)** | **12.72(29)** |
| llm_orch_qwen30b | - | - | 4.05(62) | 8.88(207) | 0.235(12) | 0.214(11) | 0.68(2) | 10.7(5) | 11.78(22) |
| llm_orch_qwen30b_reflx | - | - | **6.53(75)** | 11.49(193) | 0.357(18) | 0.334(18) | 0.46(2) | 16.7(9) | 10.07(25) |